## LangGraph 활용하기
- LangGraph는 그래프 구조를 기반으로 복잡한 AI 고급 시스템, 다중 에이전트 등을 구현하는데 특화된 라이브러리
- 그래프에는 노드(node), 엣지(edge), 상태(state)가 있음
    - `노드`: 하나의 작업 혹은 단계
    - `엣지`: 노드의 연결 단위 (직선 혹은 화살표로 시각화됨)
    - `상태`: 노드가 작업한 결과를 기록해두는 곳 (모든 노드는 상태를 공유함)
- 여러 작업을 수행하는 에이전트가 각각의 작업에 대한 워크플로우를 정의된 그래프를 통해서 수행함
- 각 단계에서 상태를 직접 제어하고 수정할 수 있어 여러 에이전트의 상호작용을 효과적으로 모델링 할 수 있음
- (LangGraph는 **각 노드의 작업이 끝날 때마다 그 결과를 chunk 객체로 반환**하며, 개발자는 이를 받아 웹or앱으로 예쁘게 쏴주거나 중간에 에러가 났는지 코드로 검사할 수 있음. 또 LangSmith 사용 시에도 노드들의 순서부터 전달된 데이터까지 모든 작업이 다 표시됨)
- LangGraph로 자율형 에이전트를 구현하려면 스스로 판단하고 작업을 수정, 보완할 수 있도록 계속 사이클이 도는 **루프(Loop) 형태**로 만들어줘야 함

In [ ]:
!pip install -q langgraph

### 1. 간단한 그래프 생성하기

In [ ]:
# StateGraph: 상태 기반 그래프 생성 클래스
# START, END : 그래프 실행이 시작되는 시점과 종료되는 시점의 특수 노드
 # START- 사용자가 처음 입력한 데이터(초기 state)를 받아 어떤 노드에서 첫번째 작업을 시작할지 그래프에게 알려주는 역할
 # END- 에이전트의 모든 작업이 끝났음을 시스템에 알리고 업데이트가 완료된 최종 상태를 사용자에게 반환. 전체 프로세스 종료하는 역할
from langgraph.graph import StateGraph, START, END

# typing: 데이터의 타입 정의 표준을 제공하는 모듈
# (사용할 데이터를 typing으로 지정해두면 전체 타입 뿐만 아니라 내용물까지 알 수 있고, 한번에 여러 타입 가능한 경우도 설정 가능.
# 최근 트렌드 AI라이브러리나 API에서는 대부분 타입 지정 방식을 표준으로 사용)
 # TypedDict: 딕셔너리의 key, value의 데이터 타입을 지정해주는 클래스
 # 정해진 key와 자료형 type만 허용함! 코드를 짤 때 오타나 잘못된 타입을 잡아줌
 # (langgraph에서 노드와 노드 사이에 데이터가 오갈 때 딕셔너리를 사용하는데, 상태의 데이터 구조를 정의할때 TypedDict를 상속받아 사용하게 됨)
from typing import TypedDict

# 그래프 도식화 클래스
from IPython.display import Image, display

In [ ]:
# step 1: 상태(State) 정의
 # langgraph의 상태는 데이터 자체가 아니라 '데이터를 담을 그릇의 모양(설계도)'를 정의하는 곳
 # 1개의 에이전트에는 1개의 전역 상태(Global State)만 존재
class MyState(TypedDict) :
    messages: str

    # messages라는 변수에는 문자열만 들어갈 수 있고, 딕셔너리로 동작(TypedDict 상속)함
    # (실제 전달되는 데이터는 {'messages': '실제 들어간 문자열'} 형태가 됨)
    # 상태에 정의하지 않은 변수(key)로는 노드로 작업할 수 없음
    # (작업 자체는 가능하지만 이를 상태에 업데이트 할 수 없다고 보면 됨!!!)


# step 2. 노드(Node) 정의: AI가 특정 작업을 수행할 노드 만들기
 # 노드는 함수로 설정하며, 노드로 등록할 함수는 반드시 state를 첫 매개변수로 받아야 함
def say_hello(state: MyState):
    # MyState에 정의된 messages 확인
    print(f'(로그) 현재 상태(state): {state['messages']}')
    
    # 노드의 반환값은 state를 업데이트하는 목적이며, 딕셔너리로 반환
    return {"messages": "Hello, I'm LangGraph!"}


# step3. 그래프 구성 (흐름도 그리기)
graph = StateGraph(MyState)           # 우리가 정의한 state로 그래프 객체 생성
graph.add_node('hello', say_hello)    # AI 노드 추가 (say_hello 노드를 'hello'로 지칭함)

graph.add_edge(START, 'hello')     # 엣지 추가(시작 > hello)
graph.add_edge('hello', END)       # 엣지 추가(hello > 끝)


# step 4. 그래프 실행
app = graph.compile()            # 설계한 그래프를 실행 가능 객체로 변환
result = app.invoke({"messages": ""})  # 그래프 실행 (빈 메시지를 첫 노드에 전달)
                                       # (상태에 정의된 message를 활용해서 작업해야 함)

print(f"app 실행 결과 반환값: {result}")
print(f"최종 상태(state): {app.invoke({'messages':'안녕??'})}")

# 현재 노드가 할 수 있는 작업은 상태에 똑같은 인삿말을 업데이트 하는 것 뿐. 어떤 messages를 입력해도 같은 인사만 출력 중.
# 입력값을 활용하고 싶다면 노드 내에서 입력값을 읽어서 처리하는 로직을 짜야 함


In [ ]:
# 그래프 시각화
display(Image(app.get_graph().draw_mermaid_png()))

### 2. 단방향 그래프 기반 대화형 모델 만들기
- **DAG(Directed Acyclic Graph)**: 순환하지 않고 한 방향으로 흐르는 노드와 엣지로 구성된 그래프 구조
- 단방향 작업이라면 Langchain의 LCEL처럼 chain으로 구성해도 되지만, 작업 중간에 실행을 멈추고 사람의 승인을 받거나 실행전후의 상태(state)를 데이터베이스에 세이브 파일로 남기는 등 세부작업을 하고 싶다면 LangGraph 써야 함

In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# Annotated: 특정 변수의 자료형 및 사용 규칙에 대한 부가설명을 붙여주는 타입 클래스
# (상태에 값을 덮어쓰지 않고 대화 기록처럼 쌓고 싶을때 사용)
from typing import Annotated

# add_messages: 메시지를 추가하여 누적해주는 클래스
 # 대화기록을 계속 저장하고 관리하기 위해서는 메시지를 누적으로 받아줘야 하는데 이때 활용
 # 또한 add_messages는 메시지의 고유ID를 검사해서 이미 있는 메시지면 내용만 업데이트하고 새로운 메시지일 때만 뒤에 추가해줌
 # 이전처럼 messages.append 할 필요 없음!!
from langgraph.graph.message import add_messages

# MemorySaver: LangGraph에서 활용하는 기억 메모리 저장소 클래스
 # Langchain의 memory 클래스는 이전 대화 내역을 다양한 방식으로 기억하지만, MemorySaver는 대화내역 뿐만 아니라
 # agent의 state를 통째로 저장하며 '지금 어떤 도구를 쓰는 중인지', '중간 계산값은 무엇인지' 까지 기억함
 # 추후 HITL를 구현할 때 일반 memory로는 구현할 수 없고 MemorySaver가 필수적임
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
load_dotenv()
MY_API_KEY = os.getenv('OPENAI_API_KEY')

In [ ]:
# llm 정의
llm = ChatOpenAI(
    api_key=MY_API_KEY,
    model='gpt-4o'
)

# 상태 정의
class ChatState(TypedDict) :
    # Annotated는 데이터의 형태를 규정하는 '타입 클래스'로 문법적으로 []를 사용함
     # add_messages는 메시지 추가 뿐 아니라 같은 ID의 메시지가 들어오면 중복 추가하지 않고 덮어쓰는 등 똑똑한 처리까지 해줌
     # (우리는 채팅 모델 만들 거니까 메시지가 누적되어야 함. add_messages는 'role':'content' 형태로 메시지 관리함)
    messages: Annotated[list, add_messages]   # [데이터의 타입, 이전 상태와의 결합방식]


# 노드 정의
def chatbot_node(state: ChatState) :
    # 현재까지 누적된 메시지 리스트를 LLM에게 전달하여 답변 생성
    # state에 있는 딕셔너리에서 messages를 불러오는 식
    response = llm.invoke(state["messages"])

    # 생성된 답변을 messages에 반환 (add_messages에 의해 기존 리스트에 자동으로 누적됨)
    return {"messages":[response]}


# 그래프 객체 생성
graph2 = StateGraph(ChatState)

# 노드 및 엣지 추가
graph2.add_node('chatbot', chatbot_node)
graph2.add_edge(START, 'chatbot')
graph2.add_edge('chatbot', END)

# 메모리 객체 생성
memory = MemorySaver()

# 그래프 실행 객체로 변환 (메모리 추가)
app2 = graph2.compile(checkpointer=memory)

In [ ]:
display(Image(app2.get_graph().draw_mermaid_png()))

In [ ]:
config = {'configurable': {'thread_id':'user_1'}}
# MemorySaver를 설정해도 config가 없으면 누구의 대화인지 몰라서 기억 못함
# 사용자가 한 명이어도 설정해줘야 함!!

print('LangGraph를 활용한 챗봇 서비스에 오신 걸 환영합니다!')
print("대화를 종료하시려면 '종료'를 입력하세요")
print('='*50)

while True :
    user_input = input('사용자\t:')

    if user_input.strip() == '종료':
        print('대화를 종료합니다!')
        break

    # 빈칸 입력 방지 (엔터만 쳤을 경우 다시 입력받도록 함)
    # 빈칸이 입력되면 API 요청 요금이 낭비되고, 메모리에 쓸데없는 기록이 찰 수 있음
    if not user_input.strip() :
        continue

    print("AI\t: ", end="") # 답변 출력 시작점
    # stream 방식으로 답변 출력
    # (msg는 AIMessageChunk 형태, metadata는 딕셔너리 형태로 반환)
    for msg, metadata in app2.stream(
        {'messages': [('user', user_input)]},
        config,
        stream_mode='messages'   # 하나의 토큰 생성마다 즉시 반환
        # 'updates': 각 노드가 실행을 마치고 state를 업데이트할 때마다 상태값 반환
        # 'values': state 전체 내용을 통째로 계속 복사해서 반환
    ):
        
        # <meta data 주요 인자들>
         # langgraph_step: thread가 처음 생성된 이후 노드 경유 횟수
         # langgraph_node: 어떤 노드에서 작업 중인지 확인
         # ls_provider: LLM의 제공자로 멀티에이전트 구현시 코드는 claude에, 챗은 gpt에 요청하는 것처럼 LLM들에게 다른 작업을 시킬 수 있음
         # langgraph_triggers: 현재 노드가 실행되도록 한 주체 확인 (디버깅에 활용)


        # # 확인용 코드
        # print("msg", msg)
        # print()
        print('metadata', metadata)
        # print()


        # msg에 내용이 들어있고 chatbot 노드에서 나온 메시지만 화면에 출력
        if msg.content and metadata.get('langgraph_node') == 'chatbot' :
            print(msg.content, end="", flush=True)

    print()

    print('<현재 상태 메모리에 저장된 내용 확인>')
    snapshot = app2.get_state(config)      # get_state: 현재상태를 가져옴 (StateSnapshot객체로 반환)
    print(snapshot.values)    # values: 현재 상태의 내용물 (messages에 HumanMessage, AIMessage가 있음)
    print()


### **3. 라우팅을 활용한 고객 클레임 처리 에이전트 만들기**
- **Routing**: 노드 간의 정적이고 확정적인 실행 흐름
- **Custom Routing**: 특정 요구사항에 맞게 라우팅 로직을 직접 정의
- **Conditional Routing**: 노드 간의 동적인 이동경로를 결정하는 제어 분리 메커니즘. Rule-based Routing, Agentic Routing으로 나뉨
    - `Rule-based Routing`: 정해진 파이썬 코드 규칙대로만 다음 노드 선택
    - `Agentic Routing`: LLM이 다음 노드를 판단하고 선택에 반영하는 방식
        - 1. 특정 노드의 작업에서 LLM이 다음 상태(state)에 대한 자신의 판단결과를 상태에 기록해둠
        - 2. 현재 그래프의 상태 데이터를 인자로 받는 '라우팅 함수'를 호출
        - 3. 라우팅 함수가 상태 값을 확인하고 반환한 결과에 따라 다음으로 실행될 노드가 실시간으로 결정
        - 간단한 루프를 도는 에이전트라면 rule-based가 더 효율적, 복잡한 상황에서의 판단이 필요하면 Agentic이 좋음

#### `규칙 기반 라우팅 (Rule-based Routing)`

In [ ]:
# 상태 정의
class ChatState(TypedDict) :
    messages: Annotated[list, add_messages]

# 노드 정의 ( 두명의 작업자 생성)
# 작업자 A : 일반적인 대답을 해주는 AI 노드
def chatbot_node(state: ChatState):
    print("[시스템] 일반 챗봇 노드가 실행되었습니당.")
    response = llm.invoke(state['messages'])
    return {'messages':[response]}      # "여기 내 작업 결과물! 이 중에서 messages라고 써진 칸에 이 데이터를 업데이트해줘!"
                                        # dict로 리턴하는 이유: 한 노드에서 여러개의 칸을 동시에 업데이트 할 수도 있기 때문에
                                        # list로 감싼느 이유: add_messages는 리스트와 리스트를 더하는 방식으로 작동!!

# 작업자 B : 환불/취소 등 민감한 문제에는 상담원과 연결시키는 노드
def human_transfer_node(state: ChatState) :
    print('[시스템] 민감어 감지! 상담원 연결 노드가 실행되었습니당.')

    # 실제 상담원 연결 대신 고정메시지로 대체
    response = """불편을 드려 죄송합니다. 환불 및 취소 관련 문의는 정확한 처리를 위해 상담원에게 연결해드리겠습니다. (연결 중...)"""
    # role을 assistant로 직접 설정 (위 llm의 응답은 assistant가 자동으로 붙어서 출력됨)
    return {'messages':[('assistant', response)]}


# 조건부 라우팅 함수
 # 해당 함수는 상태(state)를 확인한 뒤, 다음으로 이동할 노드의 이름(문자열)을 반환함
def route_to_next(state: ChatState) :
    # 가장 마지막에 들어온 메시지 확인
    last_message = state["messages"][-1]

    # 메시지의 텍스트만 추출
    user_text = last_message.content
    
    # 민감 단어가 포함되어 있는지 검사
    # (실무에서 쓰려면 정규표현식 패턴이나 좀 더 복잡한 로직을 넣어 여러 민감단어가 같이 나왔을때만 상담원 연결하는 게 좋음)
    if "환불" in user_text or "취소" in user_text:
        return "human_transfer_node"     # 함수 자체를 주는게 아니라, 노드의 이름을 알려주는 것뿐!!
    else :
        return "chatbot_node"
    

# 그래프 객체 생성
graph3 = StateGraph(ChatState)
graph3.add_node('chatbot_node', chatbot_node)
graph3.add_node('human_transfer_node', human_transfer_node)

# 조건부 라우팅 엣지 등록
graph3.add_conditional_edges(
    START,           # 시작 노드 (라우팅 함수 이전에 어떤 노드?)
    route_to_next,   # 조건부 라우팅 함수

    # 라우팅 함수를 통해 이동될 노드 ({'라우팅 함수가 반환할 문자열': '실제 연결 노드명'})
    {
        'chatbot_node': 'chatbot_node',
        'human_transfer_node':'human_transfer_node'
    }
)

# 엣지 등록
graph3.add_edge('chatbot_node', END)
graph3.add_edge('human_transfer_node', END)

memory = MemorySaver()
app3 = graph3.compile(checkpointer=memory)

config = {'configurable': {'thread_id':'customer_1'}}

print('안녕하세요 고객센터에 오신 것을 환영합니다.')
print('='*50)

while True:
    user_input = input('고객\t:')
    print(user_input)
    if user_input.strip()== '종료':
        print('대화를 종료합니다. 다음에 또 이용해주세요!')
        break
    if not user_input.strip():
        continue

    result = app3.invoke({'messages':[('user', user_input)]}, config)
    print(f"AI\t: {result['messages'][-1].content}")

In [ ]:
display(Image(app3.get_graph().draw_mermaid_png()))

# 조건부 라우팅 엣지는 점선으로 표시됨

### **4. 도구를 활용한 인터넷 검색 내용 정리 및 파일 전송 에이전트 제작 (Agentic Routing)**
- **DCG(Directed Cyclic Graph)**: 순환하는 루프가 존재하는 그래프

- **타빌리 검색**
    - 타빌리 사이트: https://www.tavily.com/
    - `TAVILY_API_KEY` 가입 시 할당
- **노션 API**
    - 노션 사이트: https://www.notion.com/ko
    - 노션 개발자 API 사이트: https://www.notion.so/profile/integrations/public
    - `NOTION_API_KEY` API 통합 편집 페이지의 '프라이빗 API 통합 시크릿'
    - `NOTION_DATABASE_ID` DB표가 있는 노션 페이지의 url의 워크스페이스 이름 뒤 '/' 부터 '?'전까지 32글자
- **슬랙 API**
    - 슬랙 앱(또는 웹페이지)에서 에이전트 실습용 워크스페이스 추가
    - 슬랙 API 사이트: https://api.slack.com/apps
    - `SLACK_BOT_TOKEN` 슬랙 앱의 'Bot User OAuth Token'
    - `SLACK_CHANNEL_ID` 메시지를 보낼 채널 ID (실습용 채널의 우상단 ... 눌러서 채널 세부정보 열기의 맨하단에 있음)

In [ ]:
!pip install tavily-python langchain-tavily slack-sdk

In [3]:
import os
import json
import requests as req
from dotenv import load_dotenv
# List : 리스트 타입을 알려주는 클래스
from typing import Annotated, TypedDict, List

# WebClient : 슬랙 API 통신 클래스
from slack_sdk import WebClient
# SlackApiError : 슬랙 전용 에러 처리기 클래스
from slack_sdk.errors import SlackApiError

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage

# 랭체인 지원 타빌리 검색 클래스
from langchain_community.tools.tavily_search import TavilySearchResults

from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

# ToolNode : LLM이 도구를 결정하면 파이썬에서 해당 도구를 실행해주는 자동 실행기
# tools_condition : 랭그래프 제공 조건부 라우팅 클래스
from langgraph.prebuilt import ToolNode, tools_condition

In [ ]:
# 환경변수 로드
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
NOTION_API_KEY = os.getenv("NOTION_API_KEY")
NOTION_DATABASE_ID = os.getenv("NOTION_DATABASE_ID")
SLACK_BOT_TOKEN = os.getenv("SLACK_BOT_TOKEN")
SLACK_CHANNEL_ID = os.getenv("SLACK_CHANNEL_ID")

if not all ([OPENAI_API_KEY, TAVILY_API_KEY, NOTION_API_KEY, NOTION_DATABASE_ID, 
             SLACK_BOT_TOKEN, SLACK_CHANNEL_ID]):
    print('경고: .env 파일에 필요한 API KEY가 모두 설정되지 않았습니당')


######### <도구 정의>

# 1. 인터넷 검색 도구 (Tavily)
 # langchain과 연동된 Taviy 클래스이므로 에이전트가 바로 활용 가능한 형태
# max_result: 반환 받을 최대 검색 결과 수
search_tool = TavilySearchResults(max_result=3)

# 2. Notion API 연동 도구
@tool
def create_notion_page(title:str, content:str) -> str:
    """검색된 정보를 바탕으로 Notion 데이터베이스에 새로운 페이지를 생성합니다.
    
    Args:
        title: 새 페이지의 제목 (검색주제)
        content: 페이지 본문에 들어갈 내용 (검색 결과 요약)
    """
    
    # 노션 api url은 고정
    url = "https://api.notion.com/v1/pages"

    headers = {
        # 사용자 인증
        "Authorization": f"Bearer {NOTION_API_KEY}",
        # 노션 API 서버에서 정해둔 데이터 형식은 JSON
        "Content-Type": "application/json",
        # 개편이 있기 전에는 모두 2022년도 규칙서를 기준으로 통신한다는 뜻
        "Notion-Version": "2022-06-28"
    }

    # Notion API 요구사항에 맞게 JSON으로 데이터 구성
    # (노션은 빈 페이지에 블록들을 하나씩 조립해서 문서 만드는 구조라 이런 JSON형식의 지침과 연동이 좋음)
     # 노션 개발자 공식 문서: https://developers.notion.com/reference/post-page
    data = {
        # parent: 노션의 어느 표(데이터베이스)에 넣을지 설정
        "parent": {'database_id': NOTION_DATABASE_ID},
        # properties: 표의 어느 칸에 넣을지 설정
        "properties":{
            "제목":{
                "title:" [{
                    "text": {"content": title}   # 제목(함수의 매개변수)
                    }] 
            }
        },
        # children: 페이지 안에 어떤 본문 내용을 넣을지 설정
        # (여러개의 문단을 한번에 넣기 위해 []로 구성)
        "children":[
            {
                # 공식 문서 참조: https://developers.notion.com/reference/block
                "object": "block",
                "type": "paragraph",     # 일반 텍스트 단락 (heading_1이면 큰 제목)
                # 단락 세부 설정
                "paragraph": {
                    # 글자 꾸미기 (굵게, 기울어짐, 하이퍼링크 등)
                    "rich_text": [{
                        "type": "text",
                        "text": {
                            # 노션의 표 하나의 셀에 2000자까지만 넣을 수 있음
                            "content": content[:1999]
                        }
                    }]
                }
            }
        ]
    }

    try :
        res = req.post(url, headers=headers, json=data)

        # HTTP 응답이 성공(200번대)이 아닐 경우 노션 서버의 에러메시지를 직접 반환
        if not res.ok :
            return f"Notion API 에러 발생! 상태코드: {res.status_code}, 상세내용: {res.text}"
        # 상태코드(status_code)를 보고 에러를 발생시키도록 설정
        # (발생하는 에러를 직접 알 수 있도록. 설정이 없으면 연결된 다른 곳에서 발생한 에러가 출력될 수 있어 디버깅 힘듦)

        res.raise_for_status()
        return f"Notion 페이지 '{title}' 생성이 완료되었습니다."
    
    except Exception as e:
        return f"Notion 페이지 생성 실패 (파이썬 에러): {e}"
    
# 3. 파일 저장 도구
@tool
def save_to_file(filename:str, content:str) -> str:
    """주어진 내용을 로컬에 TXT 파일로 저장합니다.
    
    Args:
        filename: 저장할 파일 이름(예: report.txt)
        content: 파일에 기록할 내용
    """
    try :
        with open(filename, "w", encoding='utf-8') as f :
            f.write(content)
        return f"파일 '{filename}'이 성공적으로 저장되었습니다."
    
    except Exception as e :
        return f"파일 저장 실패: {e}"

<>:53: SyntaxWarning: str indices must be integers or slices, not dict; perhaps you missed a comma?
C:\Users\agnes\AppData\Local\Temp\ipykernel_22148\3101133569.py:20: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  search_tool = TavilySearchResults(max_result=3)
C:\Users\agnes\AppData\Local\Temp\ipykernel_22148\3101133569.py:53: SyntaxWarning: str indices must be integers or slices, not dict; perhaps you missed a comma?
  "title:" [{
